# Distances 

By defintion for a vacuum 

\begin{equation}
m -  M = 5 {\rm log} \left( \frac{d}{10~{\rm pc}} \right)
\end{equation}

so if we can measure the apparent magnitude $m$ and infer the absolute magnitude $M$ we can determine the distance.

$M$ can be determined a number of ways, such as the period-luminosity relation for Cepheids or the rise-fall time of Type Ia supernovae. 


# Dust 

If we have dust along the line of sight then we need to modify this equation to 

\begin{equation}
m - M = 5 {\rm log} \left( \frac{d}{10~{\rm pc}} \right) + A
\end{equation}

where $A$ is the dust obscuration in the relevant band. In other words dust makes things fainter. 

The value of $A$ can be determined from a colour offset, since dust impacts blue wavelengths more than red wavelengths. 

For example, for the $V$-band 

\begin{equation}
A_V \simeq 3.1 E(B-V)
\end{equation}

where $E(B-V)$ is the change in the $B-V$ colour resulting from dust. $A_R \simeq 3.8 E(V-R)$ and $A_I \simeq 2.3 E(R-I)$.

<IMG SRC='Mamajek_EBV.png'>


In [ ]:
# Import various libraries 

import numpy as np
import astropy
import astropy.io.fits as fits
import astropy.io.ascii as ascii
from astropy import units as u
from astropy.table import QTable
import matplotlib.pyplot as plt
import gc
from astropy.coordinates import SkyCoord
from astroquery.gaia import Gaia


# Pleiades Example 

Photometry from https://webda.physics.muni.cz/cgi-bin/rdb_list_ref.cgi?mel022+ubv.peo+25

I have converted this data to an ASCII file - take a look at it

It has missing values and this makes it a little trickier to read than a CSV file 

I have to specify which characters in each line of text are allocated to which variables

In [ ]:
# Create lists for the parameters
Number=[]
Refer=[]
V=[]
BV=[]
UB=[]
N=[]

# Open the file 
f = open('Pleiades_UBV_photometry.txt', 'r')  # We need to re-open the file
lines = f.readlines()
f.close()

# Go through the file and get the valid data 
for line in lines:
    # print(line)
    if line[0]!='#' and line[15]!=' ':
        # print(line)
        Number.append(line[0:4])
        Refer.append(line[5:10])
        V.append(float(line[10:18]))
        if line[22]!=' ':
            BV.append(float(line[19:26]))
        else:
            BV.append(-99.99)
        if line[30]!=' ':
            UB.append(float(line[27:34]))
        else:
            UB.append(-99.99)
        N.append(line[35:39])


# Create table

There are a number of ways of bundling data into tables in Python

The example below astropy tables 

The lists we created before are put into columns of the table 

It is easier having a table rather than many seperate lists 

In [ ]:
# Create an astropy table

Pleiades = QTable([Number, Refer, V, BV, UB, N], names=('Star number', 'Reference', 'V', 'B-V', 'U-B', 'N Observations'))

print(Pleiades)         # Comment

print(Pleiades[3:8])    # Comment

In [ ]:
px=[]
py=[]
for star in Pleiades:
    if star['V']>0.0 and star['B-V']>-90.0:
        py.append(star['V'])
        px.append(star['B-V'])

plt.gca().invert_yaxis()
plt.scatter(px,py,marker=".")
plt.title('Pleiades HR diagram')
plt.xlabel('$B-V$')
plt.ylabel('$V$')
plt.gca().set_xlim([-0.4, 2.0])


# Read absolute data from Carroll and Ostlie

In [ ]:
CarrollOstlie = ascii.read('CarrollOstlie_Apendix_StellarData.csv', format='csv', fast_reader=False)

In [ ]:
print(CarrollOstlie)

In [ ]:
copx=[]
copy=[]
for star in CarrollOstlie:
    if float(star['MV'])>-10.0 and float(star['B-V'])>-90.0:
        copy.append(float(star['MV']))
        copx.append(star['B-V'])

plt.gca().invert_yaxis()
# plt.scatter(copx,copy,marker=".")
plt.plot(copx, copy)
plt.title('Carroll and Ostlie HR diagram')
plt.xlabel('$B-V$')
plt.ylabel('$M_V$')
plt.gca().set_xlim([-0.4, 2.0])


# Distance Modulus

The distance modulus, $DM$,  is just the difference between the apparent and absolute magnitude resulting from the distance

We thus want to find the combination of $DM$ (distance) and $A$ (dust) that shifts the absolute magnitudes onto Pleiades

Remember that $A_V\simeq 3.1 E(B-V)$ 

In [ ]:
# Plot the Pleiades data

px=[]
py=[]
for star in Pleiades:
    if star['V']>0.0 and star['B-V']>-90.0:
        py.append(star['V'])
        px.append(star['B-V'])

plt.gca().invert_yaxis()
plt.scatter(px,py,marker=".", label='Pleiades')
plt.title('Pleiades HR diagram')
plt.xlabel('$B-V$')
plt.ylabel('$V$')
plt.gca().set_xlim([-0.4, 2.0])

# Dust parameters 
EBV=0.10
A=3.1*EBV

# Distance Modulus 
DM=5.50

# Plot the Carroll and Ostlie data shifted to account for distance and dust
copx=[]
copy=[]
for star in CarrollOstlie:
    if float(star['MV'])>-10.0 and float(star['B-V'])>-90.0:
        copy.append(float(star['MV']+DM+A))          # Shift the data to account for distance and dust, A=3.1*E(B-V)
        copx.append(star['B-V']+EBV)                 # Shift the colours to account for dust 

# plt.scatter(copx,copy,marker=".", label='Carroll and Ostlie')
plt.plot(copx,copy, label='Carroll and Ostlie', color='C1')
plt.title('Carroll and Ostlie HR diagram')
plt.xlabel('$B-V$')
plt.ylabel('$M_V$')
plt.gca().set_xlim([-0.4, 2.0])
plt.legend()


In [ ]:
d= 10.0 * pow(10.0, 0.2*DM)
print('Distance (pc): ', d)
print('Dust E(B-V): ', EBV)

# Again, but with statistics
We want to find the statistically most probable values for `EBV` and `DM` *and* the uncertainties on those quantities. 

You might be familiar with least-squares analysis, which finds the best slope and intercept for a linear relationship between a *dependent*  and an *independent* variable. Such simplistic approaches are usually insufficient for practical work, as they don't (or may not) handle
* errors on the "dependent" variable
* errors that vary for different measurements ("heteroskedastic")
* errors on the "independent" variable
* upper limits
* non-linear relationships
etc.

We need to have a more general approach and illustrate one possible avenue with the $\chi^2$ statistic (see e.g. Bevington, 1992), defined as follows:

$\chi^2 = \sum\frac{(y_i-m_i)^2}{\sigma_{y,i}^2}$

where the $y_i\pm\sigma_{y,i}$ are the measurements (note there is no dependence on the $x_i$ values) and $m_i$ is the model expected value at the same $x_i$. 

This expression has a particularly simple form if the $y_i$ are "counts" (i.e. described by a Poisson distribution), in which case $\sigma_{y,i}^2 = y_i$.

Let's illustrate with some simulated data distributed randomly with also the *error* (magnitude) randomly distributed

In [ ]:
n=10 # number of points
m=2.5 # "true" value

# each time you execute this block you'll get a new "realisation" of the data

y = []
sigma_y = abs(np.random.normal(loc=1.3,scale=0.5,size=n)) # array of error scales

for i in range(n):
    y.append(np.random.normal(loc=m,scale=sigma_y[i])) # each "measurement" is randomly distributed about m

y, sigma_y
plt.errorbar(range(n), y, yerr=sigma_y, fmt='o')
plt.axhline(m,0,n,c='k',ls='-')
plt.ylabel('m')
plt.xlabel('measurement number')

Now we can find the best estimate of the mean value (the simplest possible model) by scanning across the range of m and calculating the chi^2 at each point

In [ ]:
m_trial = np.arange(1,4,0.01) # array of trial values

# loop over the trial values and calculate the chi^2 for each

chisq = []
for _m in m_trial:
    chisq.append( sum((y-_m)**2/sigma_y**2))
    
# show the plot
plt.plot(m_trial, chisq)
imin = np.argmin(chisq)

plt.axhline(chisq[imin]+1,ls='--')
plt.axvline(m_trial[imin],ls='--')
plt.axvline(m,c='k',ls='-')
# you may want to zoom in a little
plt.ylim(chisq[imin]-1,chisq[imin]+10)
plt.xlim(2.2, 2.8)
plt.xlabel('m')
plt.ylabel('$\chi^2$')

# parameter error is estimated from the range covering the min(chi^2)+1 contour

ltp1 = np.where(chisq < chisq[imin]+1)[0]
print ('Best estimate of m={:.3f}+/-{:.3f} (cf. with "true" value of {:.3f})'.format(
    m_trial[imin], 0.5*(m_trial[max(ltp1)]-m_trial[min(ltp1)]), m))


# Back to the Pleiades

First we identify our *B-V* as the *independent variable* and *M_V* as the *dependent variable* (strictly speaking this is wrong, because formally errors are not usually permitted on the independent variable; but it is useful as an illustration)

Next we will use scipy's `curve_fit` to *fit* the C&O line to our data.

*Might give here an example of curve fitting to some data*

Our first problem is that the C&O data is tabulated, rather than a function, which we might normally use. But we can write a *function* that interpolates between the values

In [ ]:
def func_cno(bmv, dm, ebv):
    '''
    Simple function using the tabulated C&O data to generate a function 
    
    :param x:
    :param dm:
    :param ebv:
    '''
    
    a=3.1*ebv

    if np.shape(bmv) != ():
        return np.array([func_cno(x, dm, ebv) for x in bmv])
    
    i = np.argmin(abs(bmv-(CarrollOstlie['B-V']+ebv)))

    return CarrollOstlie['MV'][i]+dm+a


def plot_pleiades():
    plt.gca().invert_yaxis()
    plt.scatter(px,py,marker=".", label='Pleiades')
    plt.title('Pleiades HR diagram')
    plt.xlabel('$B-V$')
    plt.ylabel('$V$')
    plt.gca().set_xlim([-0.4, 2.0])

plot_pleiades()

bmv_arr = np.arange(-0.1, 1.8, 0.05)
plt.plot(bmv_arr, func_cno(bmv_arr, DM, EBV), label='func_cno', color='C1')
# [func_cno(x, DM, EBV) for x in bmv_arr]

plt.legend()


In [ ]:
from scipy.optimize import curve_fit

# Initial try gets very poor values

# popt, pcov = curve_fit(func_cno, px, py)

# but we can use our initial values as estimates
# this call returns the optimal value of each parameter and the covariance matrix

popt, pcov = curve_fit(func_cno, px, py, p0=(DM, EBV))

plot_pleiades()
plt.plot(bmv_arr, func_cno(bmv_arr, popt[0], popt[1]), label='best_fit', color='C1')
plt.legend()

# the covariance matrix *may* give us an estimate of the error, but see the notes:
# https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.curve_fit.html

perr = np.sqrt(np.diag(pcov))
print ('Distance modulus error: {:.2f} <=? {:.2f}'.format(perr[0], popt[0]))

# in this case the error is not < the best fit value, so likely not reliable
# an improved model function might help

d = 10.0 * pow(10.0, 0.2*popt[0])
print('Distance (pc): {:.2f}'.format(d))
print('Dust E(B-V): {:.3f}'.format(popt[1]))
#np.sqrt(np.diag(pcov)))


# How does this compare with Gaia?

In [ ]:
rad=56.869089
decd=24.105313
print(rad,decd)
# Get coordinates in astropy format
coord = SkyCoord(ra=rad, dec=decd, unit=(u.degree, u.degree), frame='icrs')
# Match radius in degrees
radius = u.Quantity(1.0, u.deg)

# Run query and return results
# This may throw an error
try:
    Gaia.ROW_LIMIT = 2000
    j = Gaia.cone_search_async(coord, radius=radius)
    r = j.get_results()
    parallax_label, gmag_label = 'parallax', 'phot_g_mean_mag'
except:
    print ("No result from Gaia.cone_search_async, falling back to Vizier")
    from astroquery.vizier import Vizier
    vizier = Vizier()
    r = vizier.query_region(coord, radius=radius, catalog='I/355/gaiadr3', column_filters={'Gmag': '<14'})
    parallax_label, gmag_label = 'Plx', 'Gmag'


print(r)
print(r[0])

In [ ]:
r[0].columns

In [ ]:
# Plot Parallax data 
gmag_label='Gmag'
gpx=[]
gpy=[]
for star in r[0]:
    if float(star[parallax_label])>0.0 and float(star[gmag_label])>-90.0:
        gpy.append(float(star[parallax_label]))          
        gpx.append(star[gmag_label])                 

plt.scatter(gpx,gpy,marker=".", label='Gaia')
plt.title('Gaia')
plt.xlabel('$G$')
plt.ylabel('Parallax (milli-arcsec)')


We look for a cluster of stars with a common parallax, which could be the cluster members; we can estimate the distance as the inverse of the parallactic angle: